# Step 4 &mdash; Dual-Stream Model Architecture

**Goal:** classify each probe over 13 wear classes (multi-label) by fusing two complementary representations &mdash; fine-grained local tiles + a macro global view.

Three architectures were compared, all sharing the **same dual-stream design** with a different backbone:

| Architecture | Backbone family | Why included |
|--------------|----------------|--------------|
| ResNet50V2 (Dual-Stream) | Classical CNN | Stable, well-understood baseline |
| EfficientNetV2-S (Dual-Stream) | Modern CNN | Strong CNN baseline (compound scaling) |
| **SwinV2-Tiny (Dual-Stream)** | Vision Transformer | Final selected model |

## Design rationale

Wear-pattern recognition depends on **both**:
- **Local texture** &mdash; subtle features like scratches, flaking, grooving (best captured by tile crops).
- **Global structure** &mdash; the overall layout of the wear track (captured by the strip view).

A single input pyramid would force a trade-off between resolution and context. Two parallel streams sidestep this: each backbone receives a representation tuned for what it needs to see, and the features are fused before the classifier head.

## SwinV2 dual-stream (final selected model)

Two key decisions:

1. **Shared backbone.** Tile and global share the same SwinV2-Tiny weights &mdash; saves parameters and lets pretrained representations transfer to both views.
2. **Global strip &rarr; chunked & mean-pooled.** Swin expects a square 256&times;256 input but our global strip is 256&times;768. We split it into three 256&times;256 chunks along the width axis, encode each separately, then **mean-pool** the chunk features. This preserves the wide global aspect ratio without distorting it.

In [ ]:
import torch
import torch.nn as nn
import timm


class SwinV2DualStream(nn.Module):
    """Dual-stream SwinV2 backbone with concat-fusion classifier head."""

    def __init__(self, backbone_name="swinv2_tiny_window8_256",
                 num_classes=13, pretrained=True,
                 shared_backbone=True, dropout=0.2,
                 global_tile_w=256):
        super().__init__()
        self.global_tile_w = global_tile_w
        self.shared_backbone = shared_backbone

        if shared_backbone:
            self.tile_bb = timm.create_model(backbone_name, pretrained=pretrained,
                                             num_classes=0, global_pool="avg")
            self.global_bb = self.tile_bb
        else:
            self.tile_bb   = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, global_pool="avg")
            self.global_bb = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, global_pool="avg")

        feat_dim = self.tile_bb.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim * 2, feat_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, num_classes),
        )

    def _encode_global(self, global_img):
        # Swin requires square input; split wide strip into chunks and mean-pool features.
        W = global_img.shape[3]
        if W == self.global_tile_w:
            return self.global_bb(global_img)
        assert W % self.global_tile_w == 0, "global width must be divisible by tile width"
        chunks = torch.split(global_img, self.global_tile_w, dim=3)
        feats = [self.global_bb(c) for c in chunks]
        return torch.stack(feats, dim=0).mean(dim=0)

    def forward(self, tile, global_img):
        f_tile   = self.tile_bb(tile)
        f_global = self._encode_global(global_img)
        return self.head(torch.cat([f_tile, f_global], dim=1))

## CNN baselines (same dual-stream skeleton)

The CNN baselines reuse the exact same dual-stream skeleton &mdash; only the backbone changes. They take fixed-resolution inputs natively, so no chunk-and-mean-pool trick is needed for the global view.

In [ ]:
class CNNDualStream(nn.Module):
    """Dual-stream wrapper for ResNet50V2 / EfficientNetV2-S baselines."""

    def __init__(self, backbone_name, num_classes=13, pretrained=True, dropout=0.2):
        super().__init__()
        self.tile_bb   = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, global_pool="avg")
        self.global_bb = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, global_pool="avg")
        feat_dim = self.tile_bb.num_features

        self.head = nn.Sequential(
            nn.Linear(feat_dim * 2, feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, num_classes),
        )

    def forward(self, tile, global_img):
        f = torch.cat([self.tile_bb(tile), self.global_bb(global_img)], dim=1)
        return self.head(f)


# Example backbones used:
#   ResNet50V2:        "resnetv2_50"
#   EfficientNetV2-S:  "tf_efficientnetv2_s"
#   SwinV2-Tiny:       "swinv2_tiny_window8_256"  (used with SwinV2DualStream)

## Why SwinV2 won on this task

From the evaluation (see Step 6):

- **Window-based self-attention** captures fine-grained local texture &mdash; ideal for the tile stream.
- **Shifted windows** mix information across window boundaries, providing the broader contextual cues needed for the global stream without paying full-attention cost.
- Both abilities combine well with the dual-stream concat-fusion: SwinV2-Tiny achieved the best **macro-F1 (0.7505)** and **mAP (0.7805)** on the test set, while CNN baselines were close on micro-F1 but weaker on class-balanced metrics.

Macro-F1 / mAP matter more here because the dataset is **class-imbalanced and multi-label** &mdash; we want every class predicted reliably, not just the common ones.